# Retail Inventory Management System
## Notebook 1 — Bronze Layer
### Purpose
This notebook ingests raw product data from Azure Data Lake Storage Gen2 into the Bronze Delta table.
No transformations are applied — data is stored exactly as received from the source API.

###  Importing required libraries 


In [0]:
spark.sql("USE CATALOG retail_inventory")

import logging
from datetime import datetime 
import pytz 
from pyspark.sql.functions import explode, current_timestamp, lit

# Logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("bronze_notebook")

logger.info("Bronze notebook execution started")

2026-04-07 09:15:06,950 - INFO - Bronze notebook execution started


#### Build Today's File Path


In [0]:
ist=pytz.timezone('Asia/Kolkata')
today=datetime.now(ist).strftime("%Y-%m-%d")

adls_path=f"abfss://raw@inventrystoragee1010.dfs.core.windows.net/external/bronze/products_{today}.json"
print(f"Reading file for date :{today} ")

Reading file for date :2026-04-07 


####  Fetch Data from ADLS Gen2


In [0]:
from datetime import datetime
import pytz

ist = pytz.timezone('Asia/Kolkata')
today = datetime.now(ist).strftime("%Y-%m-%d")

adls_path = f"abfss://raw@inventrystoragee1010.dfs.core.windows.net/external/bronze/products_{today}.json"

raw_df = spark.read.option("multiline","true").json(adls_path)

2026-04-07 09:45:38,312 - INFO - Received command c on object id p0


#### Write to Bronze Delta Table


In [0]:
from pyspark.sql.functions import explode

try:
    if "products" in raw_df.columns:
        bronze_df = raw_df.select(explode("products").alias("p")).select("p.*")
        logger.info("Exploded products column")

    bronze_count = bronze_df.count()
    logger.info(f"Total records after explode: {bronze_count}")

    logger.info("Writing data to Bronze table: retail_inventory.bronze.raw_products")

    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("retail_inventory.bronze.raw_products")

    logger.info("Data written successfully to Bronze table")

except Exception as e:
    logger.error(f"Error in Bronze transformation or writing: {str(e)}")
    raise

2026-04-07 09:15:07,937 - INFO - Exploded products column
2026-04-07 09:15:08,182 - INFO - Total records after explode: 30
2026-04-07 09:15:08,183 - INFO - Writing data to Bronze table: retail_inventory.bronze.raw_products
2026-04-07 09:15:08,183 - INFO - Received command c on object id p0
2026-04-07 09:15:08,312 - INFO - Received command c on object id p0
2026-04-07 09:15:09,312 - INFO - Received command c on object id p0
2026-04-07 09:15:09,892 - INFO - Received command c on object id p0
2026-04-07 09:15:10,312 - INFO - Received command c on object id p0
2026-04-07 09:15:10,749 - INFO - Data written successfully to Bronze table


In [0]:
%sql
select*from retail_inventory.bronze.raw_products
limit 5;

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,9.99,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",9.84,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,14.99,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,89,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,12.99,4.36,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 4, liam.garcia@x.dummyjson.com, Liam Garcia), List(Great product!, 2025-04-30T09:41:02.053Z, 5, ruby.andrews@x.dummyjson.com, Ruby Andrews), List(Would buy again!, 2025-04-30T09:41:02.053Z, 5, clara.berry@x.dummyjson.com, Clara Berry))",Ships in 1 week,BEA-CHI-LIP-004,91,"L

In [0]:
logger.info("Bronze pipeline completed successfully")

2026-04-07 09:15:11,603 - INFO - Bronze pipeline completed successfully
